In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
import threading
from math import modf

# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [38]:
def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 4500)
    rates_frame = pd.DataFrame(rates)

    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 12)
    rates_frame['slope'] = go(rates_frame)
#     rates_frame['sma']= rates_frame['close'].rolling(window=200).mean()

    return rates_frame

In [39]:
symbol = "GBPUSD"
a= get_values(symbol)
a

,open,high,low,close,tick_volume,spread,real_volume,rsi,slope
time,,,,,,,,,
2021-01-13 05:00:00,1.36797,1.36922,1.36794,1.36877,2380,9,0,NaN,NaN
2021-01-13 06:00:00,1.36877,1.36882,1.36789,1.36790,2265,9,0,0.000000,-14.164046
2021-01-13 07:00:00,1.36790,1.36825,1.36739,1.36768,1670,9,0,0.000000,-15.320486
2021-01-13 08:00:00,1.36768,1.36803,1.36703,1.36749,2053,9,0,0.000000,-15.402681
2021-01-13 09:00:00,1.36750,1.36998,1.36738,1.36799,3377,9,0,32.697919,-9.324225
...,...,...,...,...,...,...,...,...,...
2021-10-01 12:00:00,1.34733,1.34997,1.34731,1.34972,3126,9,0,66.116678,3.591801
2021-10-01 13:00:00,1.34967,1.35251,1.34891,1.35135,3662,9,0,70.820229,5.152255
2021-10-01 14:00:00,1.35136,1.35417,1.35098,1.35401,4201,9,0,76.602431,5.994490


In [5]:
def go(a):
    g = []
    for i in range(0, len(a)):
        g.append(slope(0, a.iloc[i-5].rsi, 5, a.iloc[i].rsi))
    return g

In [6]:
def slope(x1, y1, x2, y2):
    return (y2-y1)/(x2-x1)

In [7]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [ ]:
for i in range(43, len(a)):
    print(a.iloc[i].name)
    break

In [40]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 2.0
p= []
k = 0
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
lot = 0.02
low_buy = 0.0
low_sell = 0.0
checks1 = 0
m = ""

for i in range(205, len(a)):
        if a.iloc[i].name.hour == 0:
            store = a.iloc[i].close
        if a.iloc[i].name.hour == 9 and a.iloc[i].close > a.iloc[i].open and check == 0:
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name, "buy")
            print(abs(a.iloc[i].close - a.iloc[i].open))
            
            print("*"*20)

            check = 1 
            checks = 0
            up = 0
        if a.iloc[i].name.hour == 9 and a.iloc[i].close < a.iloc[i].open and checks == 0:
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name, "sell")
            print(abs(a.iloc[i].close - a.iloc[i].open))
            print("*"*20)

            check = 0 
            checks = 1
            up = 0

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            if pp >= 2.0:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}  buy")
                check = 0
            elif a.iloc[i].close < store:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}  close_buy")
                check = 0
            if a.iloc[i].name.hour == 23:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}  buyy")
                check = 0
        elif checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            if pp >= 2.0:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}  sell")
                checks = 0
            elif a.iloc[i].close < store:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}  close_sell")
                checks = 0
            if a.iloc[i].name.hour == 23:
                profit.append(pp)
                print(f"{pp}--  {a.iloc[i].name}  selll")
                checks = 0
            
                


####################
2021-01-26 09:00:00 sell
0.0028599999999998627
********************
-5.6--  2021-01-26 10:00:00  close_sell
####################
2021-01-27 09:00:00 buy
0.0010700000000001264
********************
-8.48--  2021-01-27 13:00:00  close_buy
####################
2021-01-28 09:00:00 sell
0.0020100000000000673
********************
4.74--  2021-01-28 10:00:00  sell
####################
2021-01-29 09:00:00 sell
0.0014000000000000679
********************
1.74--  2021-01-29 10:00:00  close_sell
####################
2021-02-01 09:00:00 sell
0.00041000000000002146
********************
2.6--  2021-02-01 10:00:00  sell
####################
2021-02-02 09:00:00 sell
0.0005100000000000104
********************
2.44--  2021-02-02 14:00:00  sell
####################
2021-02-03 09:00:00 buy
0.0018700000000000383
********************
-2.36--  2021-02-03 10:00:00  close_buy
####################
2021-02-04 09:00:00 sell
0.0009299999999998754
********************
1.92--  2021-02-04 10:00:00 

####################
2021-04-30 09:00:00 sell
0.00027999999999983594
********************
3.98--  2021-04-30 10:00:00  sell
####################
2021-05-03 09:00:00 buy
0.0013100000000001444
********************
3.38--  2021-05-03 10:00:00  buy
####################
2021-05-04 09:00:00 buy
0.0006500000000002615
********************
0.0--  2021-05-04 09:00:00  close_buy
####################
2021-05-05 09:00:00 buy
0.0004999999999999449
********************
-0.8--  2021-05-05 23:00:00  buyy
####################
2021-05-06 09:00:00 sell
0.00038999999999989043
********************
-0.08--  2021-05-06 10:00:00  close_sell
####################
2021-05-07 09:00:00 buy
0.00011999999999989797
********************
3.04--  2021-05-07 10:00:00  buy
####################
2021-05-10 09:00:00 buy
0.0020599999999999508
********************
2.36--  2021-05-10 10:00:00  buy
####################
2021-05-11 09:00:00 sell
0.0017199999999999438
********************
-5.12--  2021-05-11 23:00:00  selll
########

####################
2021-08-02 09:00:00 buy
0.0007299999999998974
********************
-5.08--  2021-08-02 15:00:00  close_buy
####################
2021-08-03 09:00:00 buy
0.000500000000000167
********************
3.06--  2021-08-03 10:00:00  buy
####################
2021-08-04 09:00:00 buy
0.0016600000000002169
********************
-6.7--  2021-08-04 17:00:00  close_buy
####################
2021-08-05 09:00:00 buy
0.00038999999999989043
********************
5.2--  2021-08-05 10:00:00  buy
####################
2021-08-06 09:00:00 buy
0.0007899999999998464
********************
0.0--  2021-08-06 09:00:00  close_buy
####################
2021-08-09 09:00:00 buy
0.0012999999999998568
********************
-2.96--  2021-08-09 10:00:00  close_buy
####################
2021-08-10 09:00:00 buy
0.0003500000000000725
********************
2.7800000000000002--  2021-08-10 11:00:00  buy
####################
2021-08-11 09:00:00 sell
0.00019999999999997797
********************
1.8--  2021-08-11 10:00:0

In [41]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))

print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

-12.960000000000004
Total negative sm -->-301.82
Total negative -->62
Total positive sm -->288.85999999999984
Total positive -->115
Length 177


In [ ]:
for i in profit:
    if i <0.0:
        print(i)

In [ ]:
sum(indexB)

In [ ]:
77+72

In [ ]:
sum(profits)

In [ ]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
lot = 0.02
low_buy = 0.0
low_sell = 0.0
print(a.iloc[34].name.hour)

for i in range(20, len(a)):
    if a.iloc[i].name.hour == 0:
        y1,c = modf(round(a.iloc[i-2].sma, 6))
        y2, c = modf(round(a.iloc[i].sma, 6))
        m = (round(y2,6)*100000-round(y1,6)*100000)/2

        if m >= 5.0:                
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)

            check = 1 
            checks = 1
        if m <= -5.0:                
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)

            check = 1  
            checks = 1
    
    elif a.iloc[i].name.hour != 0:
        if check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            if pp >= 1.0:
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}   {m}  sell")
                index.append(1)
                check = 0
            if pp <=-5.0:
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}   {m}  sell")
                index.append(1)
                check = 0
        if a.iloc[i].name.hour == 15:
            if check == 1:
                sell_price = a.iloc[i].close
                pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}    {m}  selll")               
                index.append(1)
                check = 0
                
        if checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            if pp >= 1.0:
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}   {m}  buy")
                index.append(1)
                checks = 0
            if pp <=-5.0:
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}-   {m}  buy")
                index.append(1)
                checks = 0
        if a.iloc[i].name.hour == 15:
            if checks == 1:
                sell_price = a.iloc[i].close
                pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}    {m}  buyy")               
                index.append(1)
                checks = 0

In [ ]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
lot = 0.02
low_buy = 0.0
low_sell = 0.0
m = ""

for i in range(20, len(a)):
    if a.iloc[i].name.hour == 3:
        if (a.iloc[i-3].close - a.iloc[i].close) <= -0.060:                
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)

#             check = 1 
            checks = 1
        if (a.iloc[i-3].close - a.iloc[i].close) >= 0.060:                
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)

            check = 1  
#             checks = 1
    
#     elif a.iloc[i].name.hour != 0:
    else:
        if check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            if pp >= 4.0:
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}   {m}  sell")
                index.append(1)
                check = 0
#             if pp <=-5.0:
#                 profit.append(pp)
#                 print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}   {m}  sell")
#                 index.append(1)
#                 check = 0
        if a.iloc[i].name.hour == 0:
            if check == 1:
                sell_price = a.iloc[i].close
                pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}    {m}  selll")               
                index.append(1)
                check = 0
                
        if checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            if pp >= 4.0:
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}   {m}  buy")
                index.append(1)
                checks = 0
#             if pp <=-5.0:
#                 profit.append(pp)
#                 print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}-   {m}  buy")
#                 index.append(1)
#                 checks = 0
        if a.iloc[i].name.hour == 0:
            if checks == 1:
                sell_price = a.iloc[i].close
                pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
                profit.append(pp)
                print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}    {m}  buyy")               
                index.append(1)
                checks = 0

In [ ]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
lot = 0.02
low_buy = 0.0
low_sell = 0.0
m = ""

for i in range(20, len(a)):
    if a.iloc[i].name.hour == 11:
        if a.iloc[i].close < a.iloc[i].open:                
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)

#             check = 1 
            checks = 1
        if a.iloc[i].close > a.iloc[i].open:                
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)

            check = 1  
#             checks = 1
    
#     elif a.iloc[i].name.hour != 0:
    else:
        if check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            profit.append(pp)
            print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}  sell")
            index.append(1)
            check = 0
                
        if checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            profit.append(pp)
            print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}     buy")
            index.append(1)
            checks = 0

In [36]:
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()

#IF CLOck 
b = a
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 3.0
p= []
k = 0
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
lot = 0.02
low_buy = 0.0
low_sell = 0.0
checks1 = 0
m = ""

for i in range(200, len(a)):
    if a.iloc[i].name.hour == 0 and a.iloc[i].name.minute == 5:
        if check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            profit.append(pp)
            print(f"{pp}--  {round(a.iloc[i].slope,2)}--{a.iloc[i].name}  sell___")
        if checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            profit.append(pp)
            print(f"{pp}--  {round(a.iloc[i].slope,2)}--{a.iloc[i].name}     buy____")
        buy_price = a.iloc[i].close
        print("#"*20)
        print(a.iloc[i].name)
        print("*"*20)

            

        check = 1 
        checks = 1
        checks1 = 1

    else:
        if check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            if pp >=peck:
                profit.append(pp)
                print(f"{pp}--  {round(a.iloc[i].slope,2)}--{a.iloc[i].name}  sell")
                index.append(1)
                check = 0
                
        if checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            if pp >= peck:
                profit.append(pp)
                print(f"{pp}--  {round(a.iloc[i].slope,2)}--{a.iloc[i].name}     main_buy")
                index.append(1)
                checks = 0

#         if checks1 == 1:
#             sell_price = a.iloc[i].close
#             pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
#             if pp >= peck: 
#                 if a.iloc[i].name.hour >= 3 and k == 0:
#                     profit.append(pp)
#                     print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}     2nd_buy")
                    
#                     checks1 = 0
#                 else:
#                     sell_price = a.iloc[i].close
#                     pp = price_action(symbol, lot, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
#                     if a.iloc[i].name.hour == 5 or pp >=4.0:
#                         profit.append(pp)
#                         print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}     2nd_buy")
#                         checks1 = 0
#                     k = 1
#             else:
#                 if a.iloc[i].name.hour >= 10:
#                     profit.append(pp)
#                     print(f"{pp}---{a.iloc[i].high}--{a.iloc[i].name}     2nd_buy")
#                     checks1 = 0
                    
                
                    

####################
2021-09-13 00:05:00
********************
3.22--  -2.86--2021-09-13 08:05:00  sell
0.16--  -4.85--2021-09-14 00:05:00     buy____
####################
2021-09-14 00:05:00
********************
3.3--  3.48--2021-09-14 09:45:00     main_buy
3.64--  -3.56--2021-09-14 21:35:00  sell
####################
2021-09-15 00:05:00
********************
3.02--  1.33--2021-09-15 09:05:00     main_buy
-6.04--  -6.55--2021-09-16 00:05:00  sell___
####################
2021-09-16 00:05:00
********************
3.02--  0.83--2021-09-16 01:35:00     main_buy
3.42--  -2.4--2021-09-16 09:50:00  sell
####################
2021-09-17 00:05:00
********************
3.06--  0.22--2021-09-17 02:00:00     main_buy
3.98--  -2.62--2021-09-17 17:10:00  sell
####################
2021-09-20 00:05:00
********************
3.08--  -0.38--2021-09-20 01:20:00  sell
-17.7--  -3.08--2021-09-21 00:05:00     buy____
####################
2021-09-21 00:05:00
********************
3.2800000000000002--  2.61--2021-09

In [37]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))

print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

18.240000000000002
Total negative sm -->-58.5
Total negative -->6
Total positive sm -->76.74
Total positive -->24
Length 30
